# 거시경제 충격에 따른 스테이블코인 디페깅 위험 전이 분석 및 조기 경보 모델

**분석 대상**: USDC, DAI (2020-01-01 ~ 2026-03-25, 2,261일)  
**참고 논문**: Yi-Hsi Lee et al. (2025)  
**최종 모델**: v2 (텍스트 변수 미포함, Feature Selection Top 60)

---

## 목차
0. 설정 & 데이터 로드
1. 디페깅 정의 & 타깃 변수
2. Feature Selection (v2 — MI 필터 + Log1p + RF Top 60)
3. CV 전략 & 불균형 처리
4. 모델 비교 (RF / XGBoost / LightGBM / SVM)
5. 텍스트 변수 실험 (v2 vs v3 비교 → 텍스트 미포함 확정)
6. SHAP 분석 (v2)
7. 3단계 조기경보 시스템
8. 결론 & 한계

## 0. 설정 & 데이터 로드

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import warnings
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_DIR = os.path.join(os.getcwd(), "..")
ML_DIR = os.path.join(PROJECT_DIR, "data", "ml")
PROC_DIR = os.path.join(PROJECT_DIR, "data", "processed")
OUT_DIR = os.path.join(PROJECT_DIR, "outputs", "ml")

RANDOM_STATE = 42

# v2 데이터 로드
v2_data = {}
for coin in ["USDC", "DAI"]:
    X = pd.read_csv(os.path.join(ML_DIR, f"v2_X_{coin.lower()}.csv"))
    Y = pd.read_csv(os.path.join(ML_DIR, f"v2_Y_{coin.lower()}.csv"), header=None).squeeze()
    X = X.replace([np.inf, -np.inf], np.nan)
    v2_data[coin] = {"X": X, "Y": Y}
    print(f"[{coin}] X: {X.shape}, Y: {len(Y)}, 디페깅: {Y.sum():.0f}건 ({Y.mean()*100:.1f}%)")

## 1. 디페깅 정의 & 타깃 변수

### 디페깅 판별 기준 (확정)

$$TP_t = \frac{High_t + Low_t + Close_t}{3}$$

$$D_t = \begin{cases} 1 & \text{if } |TP_t - 1| > 0.01 \\ 0 & \text{otherwise} \end{cases}$$

$$Depeg_t = \begin{cases} 1 & \text{if } D_t + D_{t-1} + D_{t-2} \geq 2 \quad (\text{2-of-3 persistence filter}) \\ 0 & \text{otherwise} \end{cases}$$

- **Typical Price(TP)** 기반 — 일중 변동 반영
- **±1% 고정 임계값** — 해석 명확, 재현 용이
- **2-of-3 필터** — 일시적 노이즈 제거, 지속적 이탈만 탐지
- **Y = Depeg_{t+1}** (다음 날 디페깅 예측)

In [ ]:
# 디페깅 타임라인 시각화
fig, axes = plt.subplots(2, 1, figsize=(16, 6), sharex=True)

for ax, coin in zip(axes, ["USDC", "DAI"]):
    df = pd.read_csv(os.path.join(PROC_DIR, f"df_{coin.lower()}.csv"), parse_dates=["Date"])
    depeg_col = f"depeg_{coin}"
    
    ax.fill_between(df["Date"], df[depeg_col], alpha=0.4, color="red", label="Depeg")
    ax.set_ylabel(coin)
    ax.set_ylim(-0.05, 1.1)
    ax.legend(loc="upper right")
    
    n_depeg = df[depeg_col].sum()
    ax.set_title(f"{coin} — 디페깅 이벤트 ({n_depeg:.0f}건, {n_depeg/len(df)*100:.1f}%)")

axes[1].set_xlabel("Date")
plt.tight_layout()
plt.show()

## 2. Feature Selection (v2)

### 3단계 파이프라인 (MI 필터 + Log1p + 공선성 제거 + RF Top 60)

1. **Step1 기계적 필터**: 분산=0, 결측≥50%, **MI<0.001** 제거 (Mutual Information — 비선형 관계 포착)
2. **Step1c Log1p 변환**: 고왜도 수준값(m2_supply, fed_balance_sheet 등) → SHAP 해석성 개선
3. **Step2 다중공선성**: |r|≥0.95인 변수 쌍에서 타겟 상관 낮은 쪽 제거
4. **Step3 RF Importance**: Top 60개 선택

- **Winsorize 미적용**: 극단값 = 디페깅 핵심 신호이므로 보존
- **MI 필터 채택 이유**: Pearson |r|<0.01 필터는 비선형 변수(return_1d 등)를 누락 → MI로 교체하여 비선형 관계도 포착

In [ ]:
# v2 선택된 변수 — 카테고리별 구성
for coin in ["USDC", "DAI"]:
    cats = pd.read_csv(os.path.join(ML_DIR, f"v2_feature_categories_{coin.lower()}.csv"))
    feats = pd.read_csv(os.path.join(ML_DIR, f"v2_selected_features_{coin.lower()}.csv"))
    
    print(f"\n[{coin}] 총 {len(feats)}개 변수 — 카테고리별 구성:")
    cat_counts = cats.groupby("category").size().sort_values(ascending=False)
    for cat, cnt in cat_counts.items():
        print(f"  {cat:25s} {cnt}개")
    
    print(f"\n  RF Importance Top 10:")
    for i, row in feats.head(10).iterrows():
        print(f"    {i+1:2d}. {row['feature']}")

## 3. CV 전략 & 불균형 처리

### CV 전략: StratifiedKFold(n_splits=5, shuffle=True)

디페깅 이벤트의 97%가 2020년에 집중 → 시계열 기반 분할(TimeSeriesSplit, Purged K-Fold) 시 test fold에 양성 0~2건으로 평가 불가.

| CV 방식 | USDC F2 | DAI F2 | 비고 |
|---------|---------|--------|------|
| Purged K-Fold (embargo=7d) | 0.000 | 0.063 | 시계열 보존, 양성 편중으로 실패 |
| **StratifiedKFold (5-fold)** | **0.906** | **0.878** | 양성 균등 배분, **채택** |

> **한계**: 시계열 순서 미반영에 따른 data leakage 가능성은 연구의 한계로 명시.  
> Purged K-Fold 검증으로 이것이 leakage가 아닌 **클래스 분포의 시간적 편중**임을 확인.

### 불균형 처리: 13개 기법 체계적 비교

USDC 33건(1.4%), DAI 126건(5.5%) — 극심한 클래스 불균형  
→ 13기법 × 2모델(RF, XGB) × 2코인 = 52조합 StratifiedKFold CV 실험

In [ ]:
# 불균형 실험 결과 히트맵 (13기법 × 2모델 × 2코인)
df_imb = pd.read_csv(os.path.join(ML_DIR, "imbalance_results.csv"))

fig, axes = plt.subplots(1, 2, figsize=(18, 10))
metrics = ["f2_mean", "recall_mean", "precision_mean", "f1_mean", "auc_pr_mean", "auc_roc_mean"]
labels = ["F2", "Recall", "Precision", "F1", "AUC-PR", "AUC-ROC"]

for ax, coin in zip(axes, ["USDC", "DAI"]):
    sub = df_imb[df_imb["coin"] == coin].copy()
    sub["label"] = sub["technique"] + "+" + sub["model"]
    sub = sub.sort_values("f2_mean", ascending=False)
    pivot = sub.set_index("label")[metrics]
    pivot.columns = labels
    
    sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlOrRd",
                ax=ax, vmin=0, vmax=1, linewidths=0.5)
    ax.set_title(f"{coin} — 불균형 기법 비교 (13기법 × 2모델)")
    ax.tick_params(axis="y", labelsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "imbalance_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# v2 최적 기법 요약
with open(os.path.join(ML_DIR, "best_config.json"), "r", encoding="utf-8") as f:
    best_config = json.load(f)

print("=" * 70)
print("  v2 최적 불균형 기법 (F2 기준)")
print("=" * 70)
for coin in ["USDC", "DAI"]:
    b = best_config[coin]["best_f2"]
    print(f"\n  [{coin}] {b['technique']}+{b['model']}")
    print(f"    F2       = {b['f2']:.4f} ± {b['f2_std']:.4f}")
    print(f"    Recall   = {b['recall']:.4f}")
    print(f"    Precision= {b['precision']:.4f}")
    print(f"    AUC-PR   = {b['auc_pr']:.4f}")

print("\n" + "-" * 70)
print("  USDC: WeightOnly+XGB — 양성 33건 극소 → 합성 오버샘플링 시 과적합 → 손실함수 가중치")
print("  DAI:  ADASYN+XGB    — 양성 126건 상대적 충분 → 결정 경계 근처 합성 가능")

## 4. 모델 비교 (RF / XGBoost / LightGBM / SVM)

코인별 v2 최적 불균형 기법을 고정한 채 4개 모델 비교:
- **USDC**: WeightOnly (class_weight / scale_pos_weight)
- **DAI**: ADASYN (sampling_strategy=0.5)
- SVM만 RobustScaler 적용 (median/IQR 기반, 이상치 강건)

In [ ]:
# 모델 비교 결과
df_mc = pd.read_csv(os.path.join(ML_DIR, "model_comparison_results.csv"))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, coin in zip(axes, ["USDC", "DAI"]):
    sub = df_mc[df_mc["coin"] == coin].sort_values("f2_mean", ascending=True)
    colors = {"RF": "#4C72B0", "XGB": "#DD8452", "LightGBM": "#55A868", "SVM": "#C44E52"}
    c = [colors.get(m, "#999") for m in sub["model"]]
    
    y = range(len(sub))
    ax.barh(y, sub["f2_mean"], xerr=sub["f2_std"], color=c, alpha=0.85, capsize=3)
    ax.set_yticks(y)
    ax.set_yticklabels([f"{r['technique']}+{r['model']}" for _, r in sub.iterrows()], fontsize=9)
    ax.set_xlabel("F2 Score (mean ± std)")
    ax.set_title(f"{coin} — Model Comparison (v2)")
    ax.set_xlim(0, 1.15)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "model_comparison_f2.png"), dpi=150, bbox_inches="tight")
plt.show()

# 결과 테이블
for coin in ["USDC", "DAI"]:
    sub = df_mc[df_mc["coin"] == coin].sort_values("f2_mean", ascending=False)
    print(f"\n[{coin}]")
    for _, r in sub.iterrows():
        print(f"  {r['technique']}+{r['model']:10s} F2={r['f2_mean']:.3f}±{r['f2_std']:.3f}  "
              f"Recall={r['recall_mean']:.3f}  Prec={r['precision_mean']:.3f}  "
              f"AUC-PR={r['auc_pr_mean']:.3f}  AUC-ROC={r['auc_roc_mean']:.3f}")

print("\n→ 양쪽 모두 XGBoost가 F2 기준 최적")

## 5. 텍스트 변수 실험 (v2 vs v3 → 텍스트 미포함 확정)

텍스트 파생변수(키워드 빈도 32개 + LDA 토픽 15개 + FinBERT 임베딩 유사도 15개)를 추가한 v3와 비교.  
동일 CV·불균형 기법에서 F2 개선 여부로 텍스트 변수의 기여를 평가.

In [ ]:
# 변수 개수 실험 결과 (v2 vs 텍스트 포함)
df_exp = pd.read_csv(os.path.join(ML_DIR, "feature_count_experiment.csv"))

print("=" * 80)
print("  v2 (텍스트 없음) vs 텍스트 포함 F2 비교")
print("=" * 80)

for coin in ["USDC", "DAI"]:
    sub = df_exp[df_exp["coin"] == coin]
    v2_row = sub[sub["config"] == "v2 (텍스트 없음)"]
    top60_row = sub[sub["config"] == "Top 60"]
    
    if len(v2_row) > 0 and len(top60_row) > 0:
        v2_f2 = v2_row.iloc[0]["f2_mean"]
        t60_f2 = top60_row.iloc[0]["f2_mean"]
        diff = t60_f2 - v2_f2
        print(f"\n  [{coin}]")
        print(f"    v2 (텍스트 없음): F2 = {v2_f2:.3f}")
        print(f"    Top 60 (텍스트 포함): F2 = {t60_f2:.3f}")
        print(f"    차이: {diff:+.3f} {'← 텍스트 포함 시 오히려 하락' if diff < 0 else ''}")

print("\n" + "=" * 80)
print("  결론: 텍스트 변수 포함 시 F2 개선 없음 또는 하락")
print("  원인: 양성 샘플 부족(USDC 33건)으로 텍스트가 신호보다 노이즈로 작용")
print("  → v2 (텍스트 미포함) 최종 확정")
print("=" * 80)

## 6. SHAP 분석 (v2)

v2 최적 모델로 전체 데이터 학습 후 SHAP values 계산:
- **USDC**: WeightOnly + XGBoost → `predict(pred_contribs=True)` (XGBoost 3.x + shap 0.49 호환 이슈 회피)
- **DAI**: ADASYN + XGBoost → 동일

In [ ]:
import shap
import xgboost as xgb
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from imblearn.over_sampling import ADASYN

# v2 최적 설정
V2_CONFIGS = {
    "USDC": {"technique": "WeightOnly", "model": "XGB"},
    "DAI": {"technique": "ADASYN", "model": "XGB"},
}

shap_results = {}

for coin in ["USDC", "DAI"]:
    cfg = V2_CONFIGS[coin]
    X_df = v2_data[coin]["X"]
    Y = v2_data[coin]["Y"].values
    feat_names = X_df.columns.tolist()
    X = X_df.values
    
    # Impute
    imp = SimpleImputer(strategy="median")
    X_imp = imp.fit_transform(X)
    
    # Sampling
    n_pos = int(Y.sum())
    n_neg = int((Y == 0).sum())
    
    if cfg["technique"] == "ADASYN":
        k = min(5, n_pos - 1)
        sampler = ADASYN(sampling_strategy=0.5, n_neighbors=k, random_state=RANDOM_STATE)
        X_train, Y_train = sampler.fit_resample(X_imp, Y)
    else:  # WeightOnly
        X_train, Y_train = X_imp, Y
    
    n_pos_r = int((Y_train == 1).sum())
    n_neg_r = int((Y_train == 0).sum())
    spw = n_neg_r / n_pos_r if n_pos_r > 0 else 1
    
    model = XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=spw, eval_metric="logloss",
        random_state=RANDOM_STATE, verbosity=0, n_jobs=-1
    )
    model.fit(X_train, Y_train)
    print(f"[{coin}] {cfg['technique']}+XGB 학습 완료 ({X_imp.shape[0]}→{X_train.shape[0]}행)")
    
    # SHAP — XGBoost pred_contribs
    dmat = xgb.DMatrix(pd.DataFrame(X_imp, columns=feat_names), feature_names=feat_names)
    contribs = model.get_booster().predict(dmat, pred_contribs=True)
    sv = contribs[:, :-1]
    
    shap_results[coin] = {"shap_values": sv, "X": X_imp, "Y": Y, "feat_names": feat_names}
    
    # Top 10
    mean_abs = np.abs(sv).mean(axis=0)
    top10 = np.argsort(mean_abs)[::-1][:10]
    print(f"  Top 10 변수:")
    for rank, idx in enumerate(top10):
        print(f"    {rank+1:2d}. {feat_names[idx]:35s} |SHAP|={mean_abs[idx]:.4f}")
    print()

In [ ]:
# SHAP Summary (Beeswarm) + Bar Plot
fig, axes = plt.subplots(2, 2, figsize=(20, 16))

for col, coin in enumerate(["USDC", "DAI"]):
    sv = shap_results[coin]["shap_values"]
    X = shap_results[coin]["X"]
    fn = shap_results[coin]["feat_names"]
    cfg = V2_CONFIGS[coin]
    
    plt.sca(axes[0, col])
    shap.summary_plot(sv, X, feature_names=fn, max_display=20, show=False)
    axes[0, col].set_title(f"{coin} — SHAP Beeswarm ({cfg['technique']}+XGB)")
    
    plt.sca(axes[1, col])
    shap.summary_plot(sv, X, feature_names=fn, plot_type="bar", max_display=20, show=False)
    axes[1, col].set_title(f"{coin} — SHAP Importance (Top 20)")

plt.tight_layout()
plt.show()

In [ ]:
# Dependence Plot — 코인별 Top 5 변수
for coin in ["USDC", "DAI"]:
    sv = shap_results[coin]["shap_values"]
    X = shap_results[coin]["X"]
    fn = shap_results[coin]["feat_names"]
    
    mean_abs = np.abs(sv).mean(axis=0)
    top5 = np.argsort(mean_abs)[::-1][:5]
    
    fig, axes = plt.subplots(1, 5, figsize=(25, 4))
    for i, (ax, idx) in enumerate(zip(axes, top5)):
        shap.dependence_plot(idx, sv, X, feature_names=fn, ax=ax, show=False)
        ax.set_title(f"#{i+1} {fn[idx]}", fontsize=9)
    
    plt.suptitle(f"{coin} — Dependence Plots (Top 5)", fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# Force Plot — 디페깅 샘플 (SHAP 기여도 가장 큰 상위 5개)
for coin in ["USDC", "DAI"]:
    sv = shap_results[coin]["shap_values"]
    X = shap_results[coin]["X"]
    Y = shap_results[coin]["Y"]
    fn = shap_results[coin]["feat_names"]
    
    depeg_idx = np.where(Y == 1)[0]
    abs_sum = np.abs(sv[depeg_idx]).sum(axis=1)
    top_local = np.argsort(abs_sum)[::-1][:5]
    samples = depeg_idx[top_local]
    
    fig, axes = plt.subplots(len(samples), 1, figsize=(16, 3 * len(samples)))
    if len(samples) == 1:
        axes = [axes]
    
    for i, (ax, si) in enumerate(zip(axes, samples)):
        top_k = 10
        s = sv[si]
        tidx = np.argsort(np.abs(s))[::-1][:top_k]
        colors = ["#FF4136" if s[j] > 0 else "#2196F3" for j in tidx]
        
        ax.barh(range(top_k), s[tidx], color=colors, alpha=0.8)
        ax.set_yticks(range(top_k))
        ax.set_yticklabels([f"{fn[j]}={X[si, j]:.3f}" for j in tidx], fontsize=7)
        ax.set_xlabel("SHAP value")
        ax.set_title(f"Sample #{si} (depeg=1)", fontsize=9)
        ax.invert_yaxis()
    
    plt.suptitle(f"{coin} — Force Plot (Top Depeg Samples)", fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# USDC vs DAI SHAP 비교 (Top 15)
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, coin in zip(axes, ["USDC", "DAI"]):
    sv = shap_results[coin]["shap_values"]
    fn = shap_results[coin]["feat_names"]
    mean_abs = np.abs(sv).mean(axis=0)
    top15 = np.argsort(mean_abs)[::-1][:15]
    
    color = "#DD8452" if coin == "USDC" else "#4C72B0"
    ax.barh(range(len(top15)), mean_abs[top15], color=color, alpha=0.85)
    ax.set_yticks(range(len(top15)))
    ax.set_yticklabels([fn[i] for i in top15], fontsize=8)
    ax.set_xlabel("Mean |SHAP value|")
    cfg = V2_CONFIGS[coin]
    ax.set_title(f"{coin} — Top 15 ({cfg['technique']}+XGB)")
    ax.invert_yaxis()

plt.suptitle("USDC vs DAI — SHAP Feature Importance (v2)", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "shap_coin_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

## 7. 3단계 조기경보 시스템

OOF(Out-of-Fold) 확률 기반으로 Alert/Caution 임계값 결정:
- **Alert (경보)**: F2-score 최적화 — Recall 중시, 놓치면 안 되는 위험
- **Caution (주의)**: Youden's J index 최적화 — Sensitivity + Specificity - 1, 의학 진단 cut-off 표준 (Youden, 1950)

In [ ]:
# 조기경보 임계값 및 성능
df_thresh = pd.read_csv(os.path.join(ML_DIR, "early_warning_thresholds.csv"))

print("=" * 70)
print("  3단계 조기경보 임계값")
print("=" * 70)
for _, r in df_thresh.iterrows():
    print(f"\n  [{r['coin']}]")
    print(f"    Alert  임계값: {r['t_alert']:.3f}  (F2 최적화 → OOF F2 = {r['f2_oof']:.3f})")
    print(f"    Caution 임계값: {r['t_caution']:.4f}  (Youden's J = {r['youden_j']:.3f})")

# 경보 분포
print("\n" + "=" * 70)
print("  경보 성능")
print("=" * 70)

for coin in ["USDC", "DAI"]:
    df_ew = pd.read_csv(os.path.join(ML_DIR, f"early_warning_{coin.lower()}.csv"))
    total = len(df_ew)
    levels = df_ew["level"].value_counts()
    depeg = df_ew[df_ew["Y_actual"] == 1]
    depeg_in_alert = len(depeg[depeg["level"] == "Alert"])
    false_alert = len(df_ew[(df_ew["level"] == "Alert") & (df_ew["Y_actual"] == 0)])
    
    print(f"\n  [{coin}] (총 {total}일)")
    print(f"    Normal:  {levels.get('Normal', 0)}일 ({levels.get('Normal', 0)/total*100:.1f}%)")
    print(f"    Caution: {levels.get('Caution', 0)}일 ({levels.get('Caution', 0)/total*100:.1f}%)")
    print(f"    Alert:   {levels.get('Alert', 0)}일 ({levels.get('Alert', 0)/total*100:.1f}%)")
    print(f"    디페깅 포착률: {depeg_in_alert}/{len(depeg)} = {depeg_in_alert/len(depeg)*100:.0f}%")
    print(f"    오경보(Alert but no depeg): {false_alert}일")

In [ ]:
# 경보 타임라인 시각화 (기존 이미지 로드)
from IPython.display import Image, display

for coin in ["usdc", "dai"]:
    img_path = os.path.join(OUT_DIR, f"early_warning_{coin}.png")
    if os.path.exists(img_path):
        print(f"\n{'='*60}")
        print(f"  {coin.upper()} 경보 타임라인")
        print(f"{'='*60}")
        display(Image(filename=img_path))
    else:
        print(f"  {coin.upper()} 경보 타임라인 이미지 없음: {img_path}")

## 8. 결론 & 한계

### 목표 달성

| 목표 | 기준 | USDC | DAI | 달성 |
|------|------|------|-----|------|
| F2-score | ≥ 0.85 | 0.906 | 0.878 | O |
| Recall | ≥ 90% | 90.0% | 90.5% | O |
| 디페깅 포착률 | 100% | 100% | 100% | O |

### 핵심 발견

1. **거시경제 전이 실증**: us_10y_yield(USDC SHAP 3위, 0.710), dxy_change_7d(USDC 5위, DAI 9위) — 거시경제 충격이 스테이블코인으로 전이되는 경로를 SHAP으로 정량 확인
2. **거래량 = 최강 선행 신호**: vol_7d_USDC가 SHAP 1위(4.009) — 유동성 급변이 디페깅의 가장 강력한 선행 지표
3. **불균형 기법 선택의 중요성**: 양성 극소(USDC 33건)에서는 WeightOnly, 양성 상대적 충분(DAI 126건)에서는 ADASYN 최적 → 데이터 특성에 맞는 기법 선택 필수
4. **3단계 경보 시스템**: 포착률 100%, USDC 오경보 0건, DAI 오경보 1건

### 한계

1. **Data Leakage 가능성**: StratifiedKFold(shuffle=True)로 시계열 순서 미반영. Purged K-Fold 검증으로 근본 원인이 클래스 분포의 시간적 편중임을 확인했으나, 실제 운용 시 rolling window 기반 재학습 필요
2. **텍스트 변수 한정적 기여**: F2 개선 없음(USDC -0.006, DAI -0.013). 양성 부족으로 노이즈화. LLM 기반 감성분석 도입 시 개선 가능
3. **분석 대상 제한**: USDC, DAI만 분석 (USDT 제외). 알고리즘 스테이블코인 미포함
4. **일별 데이터**: 디페깅은 시간/분 단위 전개 → 고빈도 데이터 사용 시 성능 향상 기대